In [2]:
!pip install datasets sentence-transformers faiss-cpu ollama ragas numpy \
             langchain-ollama langchain-community langchain

In [3]:
import json
import os
import time
import random
import traceback
from datetime import datetime

import numpy as np
import faiss
import ollama
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from ragas import evaluate
from ragas.metrics import context_recall, context_precision, faithfulness
from langchain_ollama import ChatOllama
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings

print("All imports OK")

All imports OK


/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_58460/1339478698.py:14: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall, context_precision, faithfulness
/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_58460/1339478698.py:14: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_recall, context_precision, faithfulness
/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_58460/1339478698.py:14: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. 

In [4]:
# ── CONFIG
# Everything marked FIXED is identical across V0, V1, V2
# Only chunking strategy changes in V1

# Retrieval — FIXED
TOP_K          = 5

# Models — FIXED
EMBED_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL      = "llama3.2:3b"
TEMPERATURE    = 0

# Sampling — FIXED (same queries.json as V0)
N_PILOT        = 10
RANDOM_SEED    = 42

# Paths
DATA_DIR       = "data"
os.makedirs(DATA_DIR, exist_ok=True)

QUERIES_FILE   = os.path.join(DATA_DIR, "queries.json")    # shared — do not regenerate
CHUNKS_FILE    = os.path.join(DATA_DIR, "v1_chunks.json")  # V1 specific
INDEX_FILE     = os.path.join(DATA_DIR, "v1_index.faiss")  # V1 specific
RESULTS_FILE   = os.path.join(DATA_DIR, "v1_results.json") # V1 specific

# Prompt — FIXED
SYSTEM_PROMPT = (
    "You are a research assistant. Answer the question using ONLY the "
    "provided context. If the context does not contain enough information, "
    "say: I cannot answer this question from the provided context. "
    "Be concise and precise."
)

print("Config set.")
print(f"  Top-K:        {TOP_K}")
print(f"  Embed model:  {EMBED_MODEL}")
print(f"  LLM:          {LLM_MODEL}  temperature={TEMPERATURE}")
print(f"  Queries file: {QUERIES_FILE}  ← shared with V0, do not regenerate")

Config set.
  Top-K:        5
  Embed model:  sentence-transformers/all-MiniLM-L6-v2
  LLM:          llama3.2:3b  temperature=0
  Queries file: data/queries.json  ← shared with V0, do not regenerate


In [5]:
# Load QASPER — same fix as V0
print("Loading QASPER...")
ds     = load_dataset(
    "allenai/qasper",
    revision="refs/convert/parquet",
    trust_remote_code=False,
)
papers = ds["train"]
print(f"Papers loaded: {len(papers)}")

# Load embedding model
print(f"\nLoading embedding model: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
tokenizer   = embed_model.tokenizer
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

# RAGAS — local Ollama judge, no OpenAI fallback
ragas_llm = LangchainLLMWrapper(
    ChatOllama(model=LLM_MODEL, temperature=TEMPERATURE)
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBED_MODEL)
)
print("\nRAGAS LLM and embeddings configured.")

# Load saved queries — DO NOT resample
with open(QUERIES_FILE) as f:
    queries = json.load(f)
print(f"Loaded {len(queries)} queries from {QUERIES_FILE}")
print("Query types:", {q["answer_type"] for q in queries})

Loading QASPER...
Papers loaded: 888

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_58460/4228853749.py:18: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_58460/4228853749.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  HuggingFaceEmbeddings(model_name=EMBED_MODEL)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



RAGAS LLM and embeddings configured.
Loaded 10 queries from data/queries.json
Query types: {'yes_no', 'abstractive', 'extractive'}


/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_58460/4228853749.py:21: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [6]:
def chunk_paragraphs(paper_id: str, paper: dict) -> list:
    """
    V1 structure-aware chunking.
    Each paragraph in each section becomes exactly one chunk.
    Paragraph boundaries are respected — no fixed window, no overlap.

    Stores section_name and paragraph_index in metadata so V2
    can use this information to build the document graph.
    """
    chunks = []
    full_text = paper.get("full_text", {})
    if not isinstance(full_text, dict):
        return chunks

    section_names   = full_text.get("section_name", [])
    paragraph_lists = full_text.get("paragraphs", [])

    chunk_local_id = 0

    for sec_idx, (section_name, paragraphs) in enumerate(
        zip(section_names, paragraph_lists)
    ):
        if not isinstance(paragraphs, list):
            continue

        for para_idx, para_text in enumerate(paragraphs):
            if not isinstance(para_text, str):
                continue
            para_text = para_text.strip()
            if not para_text:
                continue

            chunks.append({
                "paper_id":        paper_id,
                "section_name":    section_name,
                "paragraph_index": para_idx,
                "chunk_text":      para_text,
                "chunk_local_id":  chunk_local_id,
            })
            chunk_local_id += 1

    return chunks


# Test on first paper
test_paper  = papers[0]
test_chunks = chunk_paragraphs(test_paper["id"], test_paper)

print(f"Paper: {test_paper['id']}")
print(f"Chunks produced: {len(test_chunks)}")
print()
print("First chunk:")
print(f"  section_name:    {test_chunks[0]['section_name']}")
print(f"  paragraph_index: {test_chunks[0]['paragraph_index']}")
print(f"  text (first 150 chars): '{test_chunks[0]['chunk_text'][:150]}'")
print()
print("Second chunk:")
print(f"  section_name:    {test_chunks[1]['section_name']}")
print(f"  paragraph_index: {test_chunks[1]['paragraph_index']}")
print(f"  text (first 150 chars): '{test_chunks[1]['chunk_text'][:150]}'")

Paper: 1909.00694
Chunks produced: 56

First chunk:
  section_name:    Introduction
  paragraph_index: 0
  text (first 150 chars): 'Affective events BIBREF0 are events that typically affect people in positive or negative ways. For example, getting money and playing sports are usual'

Second chunk:
  section_name:    Introduction
  paragraph_index: 1
  text (first 150 chars): 'Learning affective events is challenging because, as the examples above suggest, the polarity of an event is not necessarily predictable from its cons'


In [7]:
print("Building V1 paragraph-boundary chunks across all 888 papers...")
t0         = time.time()
all_chunks = []
chunk_id   = 0
skipped    = 0

for paper in papers:
    paper_id    = paper.get("id", "unknown")
    paper_chunks = chunk_paragraphs(paper_id, paper)

    if not paper_chunks:
        skipped += 1
        continue

    # Assign global chunk_id
    for c in paper_chunks:
        c["chunk_id"] = chunk_id
        chunk_id += 1

    all_chunks.extend(paper_chunks)

elapsed = time.time() - t0

print(f"Done in {elapsed:.1f}s")
print(f"Total chunks:          {len(all_chunks):,}")
print(f"Papers skipped:        {skipped}")
print(f"Avg chunks per paper:  {len(all_chunks)/888:.0f}")
print()

# Token length distribution
sample = all_chunks[:1000]
lengths = [
    len(tokenizer.encode(c["chunk_text"], add_special_tokens=False))
    for c in sample
]
print(f"Token length (first 1000 chunks):")
print(f"  min:  {min(lengths)}")
print(f"  max:  {max(lengths)}")
print(f"  mean: {sum(lengths)/len(lengths):.0f}")

Building V1 paragraph-boundary chunks across all 888 papers...
Done in 0.2s

Token indices sequence length is longer than the specified maximum sequence length for this model (342 > 256). Running this sequence through the model will result in indexing errors



Total chunks:          46,882
Papers skipped:        1
Avg chunks per paper:  53

Token length (first 1000 chunks):
  min:  1
  max:  888
  mean: 65


In [8]:
with open(CHUNKS_FILE, "w") as f:
    json.dump(all_chunks, f)

size_mb = os.path.getsize(CHUNKS_FILE) / (1024 * 1024)
print(f"Saved {len(all_chunks):,} chunks → {CHUNKS_FILE} ({size_mb:.1f} MB)")
print()

# Show one chunk to confirm metadata
print("Sample chunk metadata:")
print(json.dumps(all_chunks[100], indent=2))

Saved 46,882 chunks → data/v1_chunks.json (27.5 MB)

Sample chunk metadata:
{
  "paper_id": "2003.07723",
  "section_name": "Expert Annotation ::: Emotion Labels",
  "paragraph_index": 10,
  "chunk_text": "Uneasiness (found it ugly/unsettling/disturbing / frightening/distasteful): This label covers situations when one feels discomfort about the line/stanza (if the line/stanza feels distasteful/ugly, unsettling/disturbing or frightens one). The labels Ugliness and Disgust were conflated into Uneasiness, as both are seldom felt in poetry (being inadequate/too strong/high in arousal), and typically lead to Uneasiness.",
  "chunk_local_id": 44,
  "chunk_id": 100
}


In [9]:
EMBED_BATCH = 256

print(f"Embedding {len(all_chunks):,} chunks...")
texts = [c["chunk_text"] for c in all_chunks]
n     = len(texts)
t0    = time.time()

embedding_parts = []
for i in range(0, n, EMBED_BATCH):
    batch = texts[i : i + EMBED_BATCH]
    vecs  = embed_model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    embedding_parts.append(vecs)

    done    = min(i + EMBED_BATCH, n)
    elapsed = time.time() - t0
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = (n - done) / rate if rate > 0 else 0
    if done % 10000 < EMBED_BATCH or done == n:
        print(f"  [{done:,}/{n:,}]  {elapsed:.0f}s elapsed  {rate:.0f} chunks/s  ETA {eta:.0f}s")

embeddings = np.vstack(embedding_parts).astype("float32")
print(f"\nEmbedding complete. Shape: {embeddings.shape}  Time: {time.time()-t0:.1f}s")

# Build FAISS index
dim   = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
faiss.write_index(index, INDEX_FILE)

size_mb = os.path.getsize(INDEX_FILE) / (1024*1024)
print(f"\nFAISS index built and saved.")
print(f"  Vectors:    {index.ntotal:,}")
print(f"  Index size")

Embedding 46,882 chunks...
  [10,240/46,882]  79s elapsed  129 chunks/s  ETA 283s
  [20,224/46,882]  158s elapsed  128 chunks/s  ETA 208s
  [30,208/46,882]  233s elapsed  130 chunks/s  ETA 129s
  [40,192/46,882]  319s elapsed  126 chunks/s  ETA 53s
  [46,882/46,882]  376s elapsed  125 chunks/s  ETA 0s

Embedding complete. Shape: (46882, 384)  Time: 376.4s

FAISS index built and saved.
  Vectors:    46,882
  Index size


In [10]:
EMBED_BATCH = 256

print(f"Embedding {len(all_chunks):,} chunks...")
texts = [c["chunk_text"] for c in all_chunks]
n     = len(texts)
t0    = time.time()

embedding_parts = []
for i in range(0, n, EMBED_BATCH):
    batch = texts[i : i + EMBED_BATCH]
    vecs  = embed_model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    embedding_parts.append(vecs)

    done    = min(i + EMBED_BATCH, n)
    elapsed = time.time() - t0
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = (n - done) / rate if rate > 0 else 0
    if done % 10000 < EMBED_BATCH or done == n:
        print(f"  [{done:,}/{n:,}]  {elapsed:.0f}s elapsed  {rate:.0f} chunks/s  ETA {eta:.0f}s")

embeddings = np.vstack(embedding_parts).astype("float32")
print(f"\nEmbedding complete. Shape: {embeddings.shape}  Time: {time.time()-t0:.1f}s")

# Build FAISS index
dim   = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
faiss.write_index(index, INDEX_FILE)

size_mb = os.path.getsize(INDEX_FILE) / (1024*1024)
print(f"\nFAISS index built and saved.")
print(f"  Vectors:    {index.ntotal:,}")
print(f"  Index size: {size_mb:.1f} MB")
print(f"  Expected:   ~{(len(all_chunks) * 384 * 4) / (1024*1024):.0f} MB")

Embedding 46,882 chunks...
  [10,240/46,882]  103s elapsed  99 chunks/s  ETA 370s
  [20,224/46,882]  179s elapsed  113 chunks/s  ETA 236s
  [30,208/46,882]  253s elapsed  119 chunks/s  ETA 140s
  [40,192/46,882]  333s elapsed  121 chunks/s  ETA 56s
  [46,882/46,882]  386s elapsed  122 chunks/s  ETA 0s

Embedding complete. Shape: (46882, 384)  Time: 385.9s

FAISS index built and saved.
  Vectors:    46,882
  Index size: 68.7 MB
  Expected:   ~69 MB


In [11]:
def retrieve(query: str, model, index, chunks: list, top_k: int) -> list:
    """
    V1 retrieval — identical to V0.
    Pure cosine similarity, no structural filtering.
    Chunking is different but retrieval mechanism is unchanged.
    """
    q_vec = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
    distances, indices = index.search(q_vec, top_k)
    return [chunks[idx]["chunk_text"] for idx in indices[0] if idx != -1]


def generate(query: str, context_chunks: list) -> str:
    context  = "\n\n---\n\n".join(context_chunks)
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"},
        ],
        options={"temperature": TEMPERATURE},
    )
    return response["message"]["content"].strip()


def evaluate_query(question, answer, ground_truth, retrieved):
    ragas_data = Dataset.from_dict({
        "question":      [question],
        "answer":        [answer],
        "contexts":      [retrieved],
        "ground_truth":  [ground_truth],
        "ground_truths": [[ground_truth]],
    })
    result = evaluate(
        ragas_data,
        metrics=[context_recall, context_precision, faithfulness],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False,
    )

    def safe_score(key):
        val   = result[key]
        score = val[0] if isinstance(val, list) else val
        return 0.0 if (score is None or
            (isinstance(score, float) and np.isnan(score))) else float(score)

    return {
        "context_recall":    safe_score("context_recall"),
        "context_precision": safe_score("context_precision"),
        "faithfulness":      safe_score("faithfulness"),
    }


print("Functions defined.")

Functions defined.


In [12]:
test_q = queries[0]["question"]
print(f"Question: {test_q}")
print()

t0        = time.time()
retrieved = retrieve(test_q, embed_model, index, all_chunks, TOP_K)
t_retr    = time.time() - t0

print(f"Retrieved {len(retrieved)} chunks in {t_retr*1000:.1f}ms")
print()
print("Retrieved chunks — section names:")
for i, idx in enumerate(range(len(retrieved))):
    # find matching chunk to show section name
    pass

# Show section metadata of retrieved chunks
q_vec      = embed_model.encode([test_q], convert_to_numpy=True,
                normalize_embeddings=True).astype("float32")
_, indices = index.search(q_vec, TOP_K)
for i, idx in enumerate(indices[0]):
    c = all_chunks[idx]
    print(f"  [{i+1}] {c['paper_id']} | {c['section_name']} | para {c['paragraph_index']}")
    print(f"       '{c['chunk_text'][:100]}'")
print()

t0       = time.time()
generated = generate(test_q, retrieved)
t_gen    = time.time() - t0
print(f"Generated answer in {t_gen:.1f}s:")
print(f"  '{generated[:200]}'")

Question: Why is this work different from text-only UNMT?

Retrieved 5 chunks in 643.6ms

Retrieved chunks — section names:
  [1] 1904.05584 | Combining Character and Word-level Representations | para 9
       'Finally, note that word only and char only are special cases of both gating mechanisms: INLINEFORM0 '
  [2] 1610.04377 | Pre-Processing Modules | para 5
       'Text Normalization is the process of translating ad-hoc abbreviations, typographical errors, phoneti'
  [3] 1911.03648 | Pre-Processing | para 4
       'A few characters that don't affect the results are replaced by an empty string.'
  [4] 1811.11365 | Implementation Details and Baseline Models | para 4
       'UNMT-text BIBREF9 : It is a mono-modal UMT model which only utilize text data and it is pretrained o'
  [5] 1810.05320 | FastText Introduction | para 8
       'FastText is more efficient and its training is relatively fast.'

Generated answer in 9.6s:
  'I cannot answer this question from the provided context. The

In [13]:
print(f"Running V1 pilot — {N_PILOT} queries")
print("-" * 60)

results     = []
t_run_start = time.time()

for i, q in enumerate(queries):
    t_q = time.time()
    try:
        t0        = time.time()
        retrieved = retrieve(q["question"], embed_model, index, all_chunks, TOP_K)
        t_retr    = time.time() - t0

        t0        = time.time()
        generated = generate(q["question"], retrieved)
        t_gen     = time.time() - t0

        t0      = time.time()
        metrics = evaluate_query(q["question"], generated, q["answer"], retrieved)
        t_eval  = time.time() - t0

        t_total = time.time() - t_q

        results.append({
            "query_index":       i,
            "paper_id":          q["paper_id"],
            "question":          q["question"],
            "ground_truth":      q["answer"],
            "answer_type":       q["answer_type"],
            "generated_answer":  generated,
            "retrieved_chunks":  retrieved,
            "context_recall":    metrics["context_recall"],
            "context_precision": metrics["context_precision"],
            "faithfulness":      metrics["faithfulness"],
            "timing": {
                "retrieval_s":  round(t_retr, 3),
                "generation_s": round(t_gen, 3),
                "evaluation_s": round(t_eval, 3),
                "total_s":      round(t_total, 3),
            },
            "status": "success",
        })

        print(
            f"[{i+1:2}/{N_PILOT}] {q['answer_type']:12} "
            f"recall={metrics['context_recall']:.3f}  "
            f"precision={metrics['context_precision']:.3f}  "
            f"faith={metrics['faithfulness']:.3f}  "
            f"({t_total:.0f}s)"
        )

    except Exception as e:
        print(f"[{i+1:2}/{N_PILOT}] ERROR: {e}")
        traceback.print_exc()
        results.append({
            "query_index": i,
            "status":      "error",
            "error":       str(e),
            **q
        })

t_run_total = time.time() - t_run_start
print("-" * 60)
print(f"Total: {t_run_total:.0f}s  ({t_run_total/N_PILOT:.0f}s per query)")

Running V1 pilot — 10 queries
------------------------------------------------------------


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Exception raised in Job[1]: OutputParserException(Invalid json output: The provided context was not useful in arriving at the given answer. The context discusses gating mechanisms and special cases, but does not provide any information about why the work is different from text-only UNMT.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Exception raised in Job[2]: TimeoutError()


[ 1/10] extractive   recall=0.000  precision=0.000  faith=0.000  (185s)


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

...

Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[2]: TimeoutError()


[ 9/10] abstractive  recall=0.000  precision=0.000  faith=0.000  (212s)


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Exception raised in Job[1]: TimeoutError()
Exception raised in Job[2]: TimeoutError()


[10/10] extractive   recall=1.000  precision=0.000  faith=0.000  (200s)
------------------------------------------------------------
Total: 1933s  (193s per query)


In [26]:
successful = [r for r in results if r.get("status") == "success"]

def mean(key, rows):
    vals = [r[key] for r in rows if key in r]
    return round(sum(vals)/len(vals), 4) if vals else None

def mean_timing(key, rows):
    vals = [r["timing"][key] for r in rows if "timing" in r]
    return round(sum(vals)/len(vals), 1) if vals else None

print("=" * 55)
print("V1 STRUCTURE-AWARE CHUNKING — PILOT RESULTS")
print("=" * 55)
print(f"  Queries: {len(successful)}/{len(results)} successful")
print()
print(f"  Context Recall:    {mean('context_recall', successful)}")
print(f"  Context Precision: {mean('context_precision', successful)}")
print(f"  Faithfulness:      {mean('faithfulness', successful)}")
print()
print("  V0 baseline for comparison:")
print("    Context Recall:    0.475")
print("    Context Precision: 0.270")
print("    Faithfulness:      0.083")
print()
print(f"  Mean time/query: {mean_timing('total_s', successful)}s")
print("=" * 55)

# Save
output = {
    "summary": {
        "variant": "V1 — Structure-Aware Chunking",
        "total_queries": len(results),
        "successful": len(successful),
        "mean_context_recall":    mean("context_recall", successful),
        "mean_context_precision": mean("context_precision", successful),
        "mean_faithfulness":      mean("faithfulness", successful),
        "mean_total_s":           mean_timing("total_s", successful),
        "note": "Context Precision and Faithfulness unreliable — RAGAS TimeoutErrors on 3B model"
    },
    "config": {
        "variant": "V1",
        "embed_model": EMBED_MODEL,
        "llm_model": LLM_MODEL,
        "top_k": TOP_K,
        "chunking": "paragraph_boundary",
        "temperature": TEMPERATURE,
        "n_queries": len(queries),
    },
    "results": results,
}

with open(RESULTS_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved → {RESULTS_FILE}")

V1 STRUCTURE-AWARE CHUNKING — PILOT RESULTS
  Queries: 10/10 successful

  Context Recall:    0.4357
  Context Precision: 0.0
  Faithfulness:      0.0

  V0 baseline for comparison:
    Context Recall:    0.475
    Context Precision: 0.270
    Faithfulness:      0.083

  Mean time/query: 193.3s
Saved → data/v1_results.json


In [28]:
from ragas import RunConfig

run_config = RunConfig(
    timeout=180,      # seconds per LLM call — was defaulting to ~30s
    max_retries=3,    # retry on timeout before giving up
    max_wait=60,      # wait between retries
)

def evaluate_query(question, answer, ground_truth, retrieved):
    ragas_data = Dataset.from_dict({
        "question":      [question],
        "answer":        [answer],
        "contexts":      [retrieved],
        "ground_truth":  [ground_truth],
        "ground_truths": [[ground_truth]],
    })
    result = evaluate(
        ragas_data,
        metrics=[context_recall, context_precision, faithfulness],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False,
        run_config=run_config,
    )

    def safe_score(key):
        val   = result[key]
        score = val[0] if isinstance(val, list) else val
        return 0.0 if (score is None or
            (isinstance(score, float) and np.isnan(score))) else float(score)

    return {
        "context_recall":    safe_score("context_recall"),
        "context_precision": safe_score("context_precision"),
        "faithfulness":      safe_score("faithfulness"),
    }

print("evaluate_query updated with RunConfig timeout=180s.")

evaluate_query updated with RunConfig timeout=180s.


In [30]:
# Test on first query only
q = queries[0]
retrieved = retrieve(q["question"], embed_model, index, all_chunks, TOP_K)
generated = generate(q["question"], retrieved)
metrics   = evaluate_query(q["question"], generated, q["answer"], retrieved)

print(f"recall:    {metrics['context_recall']}")
print(f"precision: {metrics['context_precision']}")
print(f"faith:     {metrics['faithfulness']}")

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Exception raised in Job[1]: OutputParserException(Invalid json output: The provided context was not useful in arriving at the given answer. The context discusses gating mechanisms and special cases, but does not provide any information about why the work is different from text-only UNMT.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Exception raised in Job[2]: TimeoutError()


recall:    0.0
precision: 0.0
faith:     0.0


In [32]:
from ragas import RunConfig
import logging
logging.basicConfig(level=logging.WARNING)  # suppress noise

# Run with raise_exceptions=True on one query to see the actual failure
q         = queries[3]  # pick one with known good evidence
retrieved = retrieve(q["question"], embed_model, index, all_chunks, TOP_K)
generated = generate(q["question"], retrieved)

print(f"Question:  {q['question']}")
print(f"Answer:    {generated[:150]}")
print(f"GT:        {q['answer'][:150]}")
print()
print("Retrieved sections:")
q_vec      = embed_model.encode([q["question"]], convert_to_numpy=True,
                normalize_embeddings=True).astype("float32")
_, indices = index.search(q_vec, TOP_K)
for idx in indices[0]:
    c = all_chunks[idx]
    print(f"  {c['paper_id']} | {c['section_name']} | para {c['paragraph_index']}")
print()

# Now evaluate with raise_exceptions=True to see raw error
from datasets import Dataset
ragas_data = Dataset.from_dict({
    "question":      [q["question"]],
    "answer":        [generated],
    "contexts":      [retrieved],
    "ground_truth":  [q["answer"]],
    "ground_truths": [[q["answer"]]],
})

try:
    result = evaluate(
        ragas_data,
        metrics=[context_recall],   # test recall only first
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=True,
        run_config=run_config,
    )
    print(f"Context Recall: {result['context_recall']}")
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")

Question:  What turn out to be more important high volume or high quality data?
Answer:    I cannot answer this question from the provided context. The context discusses the importance of handling both high-volume and high-quality data in bi
GT:        only high-quality data helps

Retrieved sections:
  2003.04967 | KryptoOracle ::: Architecture | para 0
  1709.09119 | Conclusion and Future Work | para 5
  1909.12673 | Observations on the Error Landscape | para 7
  2003.05995 | Related Work | para 5
  1801.02243 | Future work | para 0



Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Context Recall: [1.0]


In [34]:
try:
    result = evaluate(
        ragas_data,
        metrics=[context_precision],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=True,
        run_config=run_config,
    )
    print(f"Context Precision: {result['context_precision']}")
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {str(e)[:300]}")

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR: OutputParserException: Invalid json output: The provided context was useful in clarifying the importance of data quality and its impact on big data architecture. The context includes key information about the growth of data volume, which inspired the adoption of a big data architecture capable of handling prediction algor
